In [23]:
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_validate
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.neural_network import MLPRegressor
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

In [24]:
FILE = 'games-players-processed/games_training_set_6.csv'

### Leitura

In [49]:
df = pd.read_csv(FILE)

C:\Users\duilio\AppData\Local\Temp\ipykernel_14608\1324869264.py:1: DtypeWarning: Columns (0: seasonYear_home_p1_home_x, 1: seasonYear_home_p2_home_x, 2: seasonYear_home_p3_home_x, 3: seasonYear_away_p1_home_x, 4: seasonYear_away_p2_home_x, 5: seasonYear_away_p3_home_x, 6: seasonYear_home_p1_away_x, 7: seasonYear_home_p2_away_x, 8: seasonYear_home_p3_away_x, 9: seasonYear_away_p1_away_x, 10: seasonYear_away_p2_away_x, 11: seasonYear_away_p3_away_x, 12: seasonYear_y, 13: refereeName_y) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(FILE)


### Pré-processamento

In [50]:
# Drop non-numeric columns
df = df.select_dtypes(exclude=['category', 'object'])

# Drop columns with more than 90% missing values
threshold = len(df) * 0.8
df = df.dropna(thresh=threshold, axis=1)

# Fill remaining missing values with the mean of each column
df = df.fillna(df.mean())

# Transpose, drop duplicate 'rows' (original columns), and transpose back
df = df.T.drop_duplicates().T 

# Add a new column 'outcome_y' based on the comparison of 'homeScoreNormalTime_y' and 'awayScoreNormalTime_y'
df['outcome_y'] = np.where(df['homeScoreNormalTime_y'] > df['awayScoreNormalTime_y'], 'home', np.where(df['homeScoreNormalTime_y'] < df['awayScoreNormalTime_y'], 'away_draw', 'away_draw'))

# Sort the DataFrame by 'startTimestamp_y' in ascending order
df = df.sort_values(by='startTimestamp_y', ascending=True)

In [51]:
# Drop columns that start with 'gameId', 'homeTeamFoundationDateTimestamp', 'homeTeamId', 'awayTeamFoundationDateTimestamp', or 'awayTeamId'
df = df.filter(regex='^(?!gameId|homeTeamFoundationDateTimestamp|homeTeamId|awayTeamFoundationDateTimestamp|awayTeamId).*$')

In [52]:
df

,startTimestamp_home_p1_home_x,tournamentId_home_p1_home_x,round_home_p1_home_x,hasPlayerStatistics_home_p1_home_x,homeScoreNormalTime_home_p1_home_x,homeScorePeriod1_home_p1_home_x,awayScoreNormalTime_home_p1_home_x,awayScorePeriod1_home_p1_home_x,refereeId_home_p1_home_x,homeBallpossession_home_p1_home_x,...,awayWontacklepercent_y,homeInterceptionwon_y,awayInterceptionwon_y,homeTotalclearance_y,awayTotalclearance_y,homeGoalkicks_y,awayGoalkicks_y,homeYellowcards_y,awayYellowcards_y,outcome_y
0,1442152800,83,25,True,3,3.0,1,0.0,243629.0,38.0,...,12.0,12.0,11.0,39.0,13.0,10.0,6.0,3.0,4.0,home
1,1442170800,83,25,True,2,1.0,0,0.0,156090.0,44.0,...,10.179195,10.553978,10.675699,17.579481,22.472395,7.0,8.0,2.0,2.0,home
2,1442442600,83,26,True,1,1.0,4,0.0,244037.0,50.0,...,10.179195,10.553978,10.675699,17.579481,22.472395,8.0,3.0,4.0,3.0,home
3,1442170800,83,25,True,1,0.0,2,1.0,165187.0,51.0,...,10.179195,10.553978,10.675699,17.579481,22.472395,6.0,8.0,1.0,3.0,home
4,1442451600,83,26,True,4,1.0,0,0.0,52484.0,44.0,...,10.179195,10.553978,10.675699,17.579481,22.472395,12.0,5.0,1.0,4.0,home
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7368,1775608200,91438,1,True,0,0.0,1,0.0,787028.0,29.0,...,12.0,9.0,9.0,36.0,23.0,16.0,4.0,4.0,0.0,away_draw
5353,1775685600,2929,1,True,1,0.0,1,1.0,243495.0,68.0,...,5.0,17.0,11.0,24.0,35.0,2.0,8.0,1.0,3.0,home
7371,1777586400,91443,3,True,2,2.0,0,0.0,787846.0,43.0,...,13.0,13.0,4.0,29.0,13.0,8.0,3.0,3.0,6.0,away_draw
7370,1775694600,91443,1,True,1,0.0,1,1.0,786086.0,64.0,...,16.0,4.0,3.0,18.0,10.0,9.0,5.0,3.0,1.0,away_draw


In [53]:
df['outcome_y'].value_counts()

outcome_y
away_draw    3886
home         3486
Name: count, dtype: int64

In [54]:
train_size = int(len(df) * 0.8)
train_data = df.iloc[:train_size]
test_data = df.iloc[train_size:]

X_train = train_data.filter(regex='_x$')
X_test = test_data.filter(regex='_x$')

#y_train = train_data['homePasses_y'] + train_data['awayPasses_y']
#y_test = test_data['homePasses_y'] + test_data['awayPasses_y']
#y_train = train_data['homeCornerkicks_y'] + train_data['awayCornerkicks_y']
#y_test = test_data['homeCornerkicks_y'] + test_data['awayCornerkicks_y']
#y_train = train_data['homeScoreNormalTime_y'] - train_data['awayScoreNormalTime_y']
#y_test = test_data['homeScoreNormalTime_y'] - test_data['awayScoreNormalTime_y']
y_train = train_data['outcome_y']
y_test = test_data['outcome_y']

#scaler = StandardScaler().set_output(transform="pandas")
#X_train = scaler.fit_transform(X_train)
#X_test = scaler.transform(X_test)

In [ ]:
model = RandomForestRegressor(n_estimators=5000, max_features='sqrt', random_state=42, verbose=1, n_jobs=8)
model.fit(X_train, y_train)

In [ ]:
model = MLPRegressor(hidden_layer_sizes=(1024, 512, 256, 128, 64, 32), activation='relu', solver='adam', learning_rate='adaptive', alpha=0.001, early_stopping=False, tol=0.0001, max_iter=30, verbose=1)
model.fit(X_train, y_train)

In [ ]:
# 5. Make predictions
y_pred = model.predict(X_test)

# 6. Evaluate the model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(mse, r2)

print(y_test, y_pred)

In [ ]:
# Assume 'model' is your trained model and 'X' is your training data
importances = model.feature_importances_
feature_names = X_train.columns

# Create a DataFrame for easy sorting
feature_imp_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})

# Sort and print
ordered_imp = feature_imp_df.sort_values(by='Importance', ascending=False)
print(ordered_imp[:50])

In [ ]:
rf = RandomForestRegressor()
random_grid = {
    'n_estimators': [1000, 2000, 3000],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}
rf_random = GridSearchCV(estimator=rf, param_grid=random_grid, verbose=3, n_jobs=8)
rf_random.fit(X_train, y_train)
print(rf_random.best_params_)

In [56]:
model = RandomForestClassifier(n_estimators=10000, max_features='sqrt', random_state=42, verbose=1, n_jobs=8)
model.fit(X_train, y_train)

[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    1.0s
[Parallel(n_jobs=8)]: Done 184 tasks      | elapsed:    4.8s
[Parallel(n_jobs=8)]: Done 434 tasks      | elapsed:   11.1s
[Parallel(n_jobs=8)]: Done 784 tasks      | elapsed:   19.2s
[Parallel(n_jobs=8)]: Done 1234 tasks      | elapsed:   29.0s
[Parallel(n_jobs=8)]: Done 1784 tasks      | elapsed:   41.3s
[Parallel(n_jobs=8)]: Done 2434 tasks      | elapsed:   55.0s
[Parallel(n_jobs=8)]: Done 3184 tasks      | elapsed:  1.2min
[Parallel(n_jobs=8)]: Done 4034 tasks      | elapsed:  1.5min
[Parallel(n_jobs=8)]: Done 4984 tasks      | elapsed:  2.0min
[Parallel(n_jobs=8)]: Done 6034 tasks      | elapsed:  2.5min
[Parallel(n_jobs=8)]: Done 7184 tasks      | elapsed:  2.9min
[Parallel(n_jobs=8)]: Done 8434 tasks      | elapsed:  3.4min
[Parallel(n_jobs=8)]: Done 9784 tasks      | elapsed:  4.2min
[Parallel(n_jobs=8)]: Done 10000 out of 10000 | elapsed:

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",10000
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metr

In [47]:
def predict_with_ambiguity(model, X, threshold=0.005, ambiguous_class=1):
    probs = model.predict_proba(X)
    sorted_indices = np.argsort(probs, axis=1)
    top2_indices = sorted_indices[:, -2:]
    top_probs = np.take_along_axis(probs, top2_indices, axis=1)
    diff = top_probs[:, 1] - top_probs[:, 0]
    predictions = np.argmax(probs, axis=1)
    predictions[diff <= threshold] = ambiguous_class
    return model.classes_[predictions]

In [57]:
# Predict on test data
#y_pred = predict_with_ambiguity(model, X_test)
y_pred = model.predict(X_test)
print(pd.Series(y_pred).value_counts())

print(y_test)
print(y_pred)

# Evaluate
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
print(classification_report(y_test, y_pred))

[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 184 tasks      | elapsed:    0.2s
[Parallel(n_jobs=8)]: Done 434 tasks      | elapsed:    0.5s
[Parallel(n_jobs=8)]: Done 784 tasks      | elapsed:    0.9s
[Parallel(n_jobs=8)]: Done 1234 tasks      | elapsed:    1.4s
[Parallel(n_jobs=8)]: Done 1784 tasks      | elapsed:    2.0s
[Parallel(n_jobs=8)]: Done 2434 tasks      | elapsed:    3.1s
[Parallel(n_jobs=8)]: Done 3184 tasks      | elapsed:    4.1s
[Parallel(n_jobs=8)]: Done 4034 tasks      | elapsed:    5.1s
[Parallel(n_jobs=8)]: Done 4984 tasks      | elapsed:    6.2s
[Parallel(n_jobs=8)]: Done 6034 tasks      | elapsed:    7.4s
[Parallel(n_jobs=8)]: Done 7184 tasks      | elapsed:    8.7s
[Parallel(n_jobs=8)]: Done 8434 tasks      | elapsed:   10.3s
[Parallel(n_jobs=8)]: Done 9784 tasks      | elapsed:   12.2s


away_draw    921
home         554
Name: count, dtype: int64
2934         home
2936         home
2935    away_draw
2937    away_draw
2938         home
          ...    
7368    away_draw
5353         home
7371    away_draw
7370    away_draw
5354    away_draw
Name: outcome_y, Length: 1475, dtype: str
['away_draw' 'away_draw' 'away_draw' ... 'away_draw' 'away_draw' 'home']
Accuracy: 0.5871186440677966
              precision    recall  f1-score   support

   away_draw       0.60      0.70      0.64       786
        home       0.57      0.46      0.51       689

    accuracy                           0.59      1475
   macro avg       0.58      0.58      0.58      1475
weighted avg       0.58      0.59      0.58      1475



[Parallel(n_jobs=8)]: Done 10000 out of 10000 | elapsed:   12.4s finished


In [58]:
# Assume 'model' is your trained model and 'X' is your training data
importances = model.feature_importances_
feature_names = X_train.columns

# Create a DataFrame for easy sorting
feature_imp_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})

# Sort and print
ordered_imp = feature_imp_df.sort_values(by='Importance', ascending=False)
print(ordered_imp[:60])

                                  Feature  Importance
48   awayFinalthirdentries_home_p1_home_x    0.003155
115     homeAccuratepasses_home_p2_home_x    0.002882
814             awayPasses_away_p3_away_x    0.002707
93              homePasses_home_p2_home_x    0.002664
836     awayAccuratepasses_away_p3_away_x    0.002598
381             homePasses_away_p3_home_x    0.002535
403     homeAccuratepasses_away_p3_home_x    0.002456
835     homeAccuratepasses_away_p3_away_x    0.002453
259     homeAccuratepasses_away_p1_home_x    0.002358
813             homePasses_away_p3_away_x    0.002348
43      homeAccuratepasses_home_p1_home_x    0.002346
237             homePasses_away_p1_home_x    0.002338
764     awayAccuratepasses_away_p2_away_x    0.002331
741             homePasses_away_p2_away_x    0.002319
475     homeAccuratepasses_home_p1_away_x    0.002317
453             homePasses_home_p1_away_x    0.002304
763     homeAccuratepasses_away_p2_away_x    0.002280
801     homeBallpossession_a

In [63]:
probs = model.predict_proba(X_test)

sorted_probs = sorted(probs, key=lambda x: abs(x[0] - x[1]), reverse=True)

[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.1s
[Parallel(n_jobs=8)]: Done 184 tasks      | elapsed:    0.3s
[Parallel(n_jobs=8)]: Done 434 tasks      | elapsed:    0.9s
[Parallel(n_jobs=8)]: Done 784 tasks      | elapsed:    1.6s
[Parallel(n_jobs=8)]: Done 1234 tasks      | elapsed:    2.7s
[Parallel(n_jobs=8)]: Done 1784 tasks      | elapsed:    3.6s
[Parallel(n_jobs=8)]: Done 2434 tasks      | elapsed:    4.4s
[Parallel(n_jobs=8)]: Done 3184 tasks      | elapsed:    5.6s
[Parallel(n_jobs=8)]: Done 4034 tasks      | elapsed:    7.0s
[Parallel(n_jobs=8)]: Done 4984 tasks      | elapsed:    8.7s
[Parallel(n_jobs=8)]: Done 6034 tasks      | elapsed:   10.2s
[Parallel(n_jobs=8)]: Done 7184 tasks      | elapsed:   11.6s
[Parallel(n_jobs=8)]: Done 8434 tasks      | elapsed:   13.2s
[Parallel(n_jobs=8)]: Done 9784 tasks      | elapsed:   14.8s
[Parallel(n_jobs=8)]: Done 10000 out of 10000 | elapsed:

In [90]:
ctotal = 0
ccorrect = 0
ccorrectaway = 0
ccorrecthome = 0
for i, prob in enumerate(probs):
    if abs(prob[1] - prob[0]) >= 0.185:
        ctotal += 1
        if y_test.iloc[i] == model.classes_[np.argmax(prob)]:
            ccorrect += 1
            ccorrectaway += 1 if y_test.iloc[i] == 'away_draw' else 0
            ccorrecthome += 1 if y_test.iloc[i] == 'home' else 0

print(f"Confident predictions (as a percentage of all predictions): {ctotal/len(probs)*100:.2f}%")
print(f"Correct confident predictions: {ccorrect}")
print(f"Total confident predictions: {ctotal}")
print(f"Confidence Accuracy: {ccorrect/ctotal*100:.2f}%")
print(f"Correct away_draw predictions: {ccorrectaway}")
print(f"Correct home predictions: {ccorrecthome}")

Confident predictions (as a percentage of all predictions): 10.24%
Correct confident predictions: 108
Total confident predictions: 151
Confidence Accuracy: 71.52%
Correct away_draw predictions: 91
Correct home predictions: 17
